# Training scRepresenter

This step involves training both scNET and scGPT, and combining the resulting output. GPU usage is highly recommended.

In [1]:
#required imports and misc.

import os
import scanpy as sc
if os.path.basename(os.getcwd()) == 'scripts':
    os.chdir('..')
from src.main import run_scRepresenter

RUN_NAME = "pbmc3k"

SAVE_DIR = "output/" + RUN_NAME
os.makedirs(SAVE_DIR, exist_ok=True)

Global seed set to 0


## Prepare input data

There are two required inputs to run the scRepresenter pipeline, a scRNA-seq dataset and a gene similarity network. We already processed a scRNA-seq dataset (see the Preprocessing notebook), so we just need to read the file into memory.

For the gene network, we will use the Protein-Protein Interactions graph that we provide. For a guide on how to produce your custom network, see the Networks Section in the repository.

In [2]:
obj = sc.read("./data/pbmc3k_final.h5ad")
graph_file = 'PPI.csv'

## Model training

Optionally, the parameters of each model can be individually changed. Additionally, if no parameters are given for a given model it simply runs with the default values.

In [3]:
parameters_scnet = dict(
    annotation_file=graph_file,
    pre_processing_flag=False,
    human_flag=False,
    number_of_batches=1,
    split_cells=True,
    max_epoch=300,
    model_name=RUN_NAME,
    clf_loss=False,
)

parameters_scgpt = dict(
    model_name=RUN_NAME,
    seed=0,
    dataset_name=RUN_NAME,
    do_train=True,
    load_model="./src/scgpt/checkpoints/scGPT_human",
    mask_ratio=0,
    epochs=10,
    n_bins=51,
    MVC=False, # Masked value prediction for cell embedding
    ecs_thres=0.0, # Elastic cell similarity objective, 0.0 to 1.0, 0.0 to disable
    dab_weight=0.0,
    lr=5e-5,
    batch_size=20,
    layer_size=75,
    nlayers=4,  # number of nn.TransformerEncoderLayer in nn.TransformerEncoder
    nhead=4,  # number of heads in nn.MultiheadAttention
    dropout=0.3,  # dropout probability
    schedule_ratio=0.9,  # ratio of epochs for learning rate schedule
    save_eval_interval=5,
    fast_transformer=True,
    pre_norm=False,
    amp=True,  # Automatic Mixed Precision
    include_zero_gene = False,
    freeze = False, #freeze
    DSBN = False,  # Domain-spec batchnorm
)

Finally, you can train the models by calling the following function. There is the option of training both models, or just one. In this example we run scNET for 300 epochs and scGPT for 10.

In [4]:
common_scnet_embs, common_scgpt_embs, avg_combined_embs, conq_combined_embs, common_labels = \
    run_scRepresenter(RUN_NAME, obj, graph_file, SAVE_DIR, 1, 10)

         Falling back to preprocessing with `sc.pp.pca` and default params.
/home/guilherme/scRepresenter/src/scNET/../networks/PPI.csv
N genes: (690, 2638)
Highly variable genes: 690


Training: 100%|██████████| 1/1 [00:01<00:00,  1.43s/it]

Epoch [1/1]
  Row Loss: 1.7532
  Column Loss: 1.9981
  Classifier Loss: 0.0000
  Final Loss: 5.2922
  AUC: 0.5000
Best Network AUC: 0.5


save to output/pbmc3k/scGPT
scGPT - INFO - match 1803/2000 genes in vocabulary of size 60697.
scGPT - INFO - Resume model from src/scgpt/checkpoints/scGPT_human/best_model.pt, the model args will override the config src/scgpt/checkpoints/scGPT_human/args.json.
scGPT - INFO - Normalizing total counts ...
scGPT - INFO - Binning data ...
scGPT - INFO - Normalizing total counts ...
scGPT - INFO - Binning data ...
scGPT - INFO - train set number of samples: 1423, 
	 feature length: 392
scGPT - INFO - valid set number of samples: 159, 
	 feature length: 379
aaaaaaaaaaaaa cuda
scGPT - INFO - Total Pre freeze Params 51335689
scGPT - INFO - Total Post freeze Params 51335689
random masking at epoch   1, ratio of masked values in train:  0.0000


KeyboardInterrupt: 